# The Pretraining Loop: Optimizer, Schedule, and Mixed Precision

Building a language model requires assembling several training components that must interact correctly. In this notebook we cover the optimizer, learning rate schedule, mixed precision, gradient accumulation, and compute budgeting — the engineering pieces that turn a model and data pipeline into a functioning training system. Each topic connects directly to the [architecture from Notebook 01](/courses/llm/01-gpt-architecture.html) and the [data pipeline from Notebook 03](/courses/llm/03-data-pipelines.html).

Each component can be treated in isolation, but subtle interactions between them cause silent failures that are only discovered when loss diverges or throughput is unexpectedly slow: the `zero_grad` placement bug in gradient accumulation, the FP16 overflow that `GradScaler` masks but BF16 eliminates entirely, and the Adam bias correction that amplifies the first gradient 10× before warmup stabilizes it.

By the end, we have a complete training step function and a Chinchilla scaling law calculator that predicts whether a given hardware budget will produce a compute-optimal model.

## SGD — The Baseline

The simplest optimizer is **stochastic gradient descent** (SGD). At each step, we compute the gradient $g_t = \nabla_\theta \mathcal{L}$ on a mini-batch and take a step opposite to it:

$$\theta_{t+1} = \theta_t - \eta \cdot g_t$$

The problem in practice is that the loss landscape of a transformer is severely **ill-conditioned**: different parameters have curvatures that differ by orders of magnitude. Consider the simple 2D quadratic:

$$\mathcal{L}(\theta_1, \theta_2) = \theta_1^2 + 100\,\theta_2^2$$

The gradient is $(2\theta_1,\, 200\theta_2)$: the curvature in the $\theta_2$ direction is 100× larger. Any learning rate large enough to make progress along $\theta_1$ will cause oscillation along $\theta_2$, and any rate small enough to stabilize $\theta_2$ will make $\theta_1$ converge agonizingly slowly.

In [ ]:
#| code-fold: true
import numpy as np
import matplotlib.pyplot as plt

theta = np.array([10.0, 1.0])
lr    = 0.009  # largest stable lr for θ₂

path_sgd = [theta.copy()]
for _ in range(100):
    grad  = np.array([2 * theta[0], 200 * theta[1]])
    theta = theta - lr * grad
    path_sgd.append(theta.copy())

path_sgd = np.array(path_sgd)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(path_sgd[:, 0], label='θ₁')
axes[0].plot(path_sgd[:, 1], label='θ₂')
axes[0].set(title='SGD: parameter trajectories', xlabel='step')
axes[0].legend()
axes[1].plot(path_sgd[:, 0], path_sgd[:, 1], '-o', markersize=2)
axes[1].set(title='SGD: trajectory in parameter space', xlabel='θ₁', ylabel='θ₂')
plt.tight_layout(); plt.show()

**Figure.** The zigzagging in the right panel wastes gradient evaluations. In a 100M-parameter transformer, the condition number of the loss Hessian is orders of magnitude larger than 100.

### SGD with Momentum

Momentum partially addresses oscillation by maintaining a running velocity $v_t$:

$$v_t = \mu \cdot v_{t-1} + g_t, \quad \theta_{t+1} = \theta_t - \eta \cdot v_t$$

The accumulated velocity dampens oscillation by averaging gradients across steps. But momentum still fails to address the deeper issue: different parameters need [*different*]{.underline} learning rates. Embedding parameters covering a large vocabulary need small updates; output projection parameters need larger ones early in training. **Per-parameter adaptive learning rates** are required — exactly what Adam provides.

## Adam — Adaptive Moment Estimation

Adam maintains per-parameter first (gradient direction) and second (gradient magnitude) moment estimates with bias correction:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}, \quad \theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

The denominator $\sqrt{\hat{v}_t}$ gives each parameter its own effective learning rate. What the update rule actually does:

- If $g_t$ has been **consistently large** (high $\hat{v}_t$): step size shrinks — Adam is cautious in high-curvature directions.
- If $g_t$ has been **small or noisy** (low $\hat{v}_t$): step size grows — Adam moves faster in flat directions.

This self-tuning is why Adam dominates for training transformers.

### Hyperparameter defaults

| Hyperparameter | Default | What it controls |
|---|---|---|
| `lr` ($\eta$) | 1e-3 | Overall step size magnitude |
| `beta1` ($\beta_1$) | 0.9 | Momentum of gradient direction |
| `beta2` ($\beta_2$) | 0.999 | Momentum of gradient magnitude |
| `eps` ($\varepsilon$) | 1e-8 | Numerical stability |

[For language model training, `beta2=0.95` (GPT-3/NanoGPT default) is often preferred over `0.999`.]{.mark} The second moment memory length is $\approx 1/(1-\beta_2)$ steps: `0.999` retains ~1000 steps while `0.95` retains only ~20. Longer pretraining runs where gradient magnitudes shift significantly benefit from the shorter memory.

In [ ]:
import torch
import torch.nn as nn

# Standard Adam for language models
# (model defined in 01-gpt-architecture, used here for illustration)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),   # note: 0.95 not 0.999 for LLMs
    eps=1e-8,
)

## AdamW — Decoupled Weight Decay

L2 regularization adds a penalty $\frac{\lambda}{2}\|\theta\|^2$ to the loss, which adds $\lambda\theta$ to the gradient. In standard Adam, this regularization gradient passes through the adaptive denominator $\sqrt{\hat{v}_t}$: parameters with large gradients (high $\hat{v}_t$) have the regularization suppressed exactly where it matters most. The coupling undermines the intent of regularization.

**AdamW** fixes this with a decoupled update: the weight decay step is applied directly to the parameters, bypassing the adaptive scaling:

$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} - \eta \lambda \theta_t$$

[The weight decay term $\eta\lambda\theta_t$ is pure **shrinkage**]{.mark} — every parameter experiences the same regularization pressure, regardless of gradient history.

### Which parameters to decay

Not all parameters should be weight-decayed:

- **Decayed:** weight matrices (2D+ tensors — `nn.Linear` weights, attention projections, FFN weights). These benefit from regularization that discourages large weights.
- **Not decayed:** bias terms, RMSNorm scale parameters, embeddings (1-D tensors). Regularizing 1-D parameters shrinks them toward zero indiscriminately, which hurts.

We construct separate parameter groups to enforce this:

In [ ]:
def make_optimizer(model, lr, weight_decay, betas=(0.9, 0.95)):
    """Separate parameter groups: decay weights, don't decay biases/norms."""
    decay_params, no_decay_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2:          # weight matrices
            decay_params.append(p)
        else:                    # biases, RMSNorm params (1-D tensors)
            no_decay_params.append(p)

    param_groups = [
        {'params': decay_params,    'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ]
    return torch.optim.AdamW(param_groups, lr=lr, betas=betas)

## Batch Size for Language Models

The optimal batch size is not arbitrary. The **critical batch size** $B^* = \text{tr}(\boldsymbol{\Sigma}) / \|\boldsymbol{g}_\text{true}\|^2$ is the point of maximum gradient quality per FLOP: below $B^*$ you are wasting compute on noise; above $B^*$ you are paying for redundant gradient estimates. For LLM pretraining, $B^*$ falls in the range of 100K–500K tokens per step.

Two important relationships follow:

**Linear scaling rule:** doubling $B$ allows doubling $\eta$ while maintaining the same training dynamics, valid up to $B^*$. This is why large-batch training with a scaled-up LR often matches small-batch training in fewer steps.

**Generalization:** smaller batches find flatter minima and generalize better. For fine-tuning on small datasets, batch sizes of 8–32 often outperform larger ones — the gradient noise is a feature, not a bug.

The token-level batch size that determines actual compute throughput is:

$$\text{tokens per step} = \text{batch\_size} \times \text{seq\_len} \times \text{accum\_steps} \times n_\text{GPU}$$

Typical pretraining target: 256K–2M tokens per step, achieved via gradient accumulation when GPU memory limits the physical batch size.

In [ ]:
#| code-fold: true
# Linear scaling rule example
base_batch_size = 256
base_lr = 3e-4
new_batch_size = 1024
scaled_lr = base_lr * (new_batch_size / base_batch_size)
print(f"Scaled LR: {scaled_lr:.4f}")   # 1.2e-3

# Token batch size calculation
batch_size   = 8          # samples per GPU
seq_len      = 1024       # tokens per sample
accum_steps  = 4          # gradient accumulation steps
num_gpus     = 1
tokens_per_step = batch_size * seq_len * accum_steps * num_gpus
print(f"Tokens per step: {tokens_per_step:,}")   # 32,768

## Learning Rate — Range Test and Practical Defaults

The **LR range test** finds the right order of magnitude for any model-dataset pair without grid search. We exponentially sweep the learning rate from $10^{-7}$ to $10^{-1}$ over 100–200 steps and plot smoothed loss against LR. The optimal peak LR is slightly to the left of where the loss begins rising — the steep descent region just before divergence.

We run the test on a copy of the model so the original weights are untouched:

In [ ]:
#| code-fold: true
import math
from copy import deepcopy

def lr_range_test(
    model,
    dataloader,
    criterion,
    min_lr: float = 1e-7,
    max_lr: float = 0.1,
    num_steps: int = 150,
):
    model_copy = deepcopy(model)
    optimizer  = torch.optim.AdamW(model_copy.parameters(), lr=min_lr,
                                   betas=(0.9, 0.95))
    lrs, losses = [], []
    ratio = (max_lr / min_lr) ** (1 / num_steps)
    data_iter = iter(dataloader)
    for step in range(num_steps):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            x, y = next(data_iter)

        optimizer.zero_grad()
        logits = model_copy(x)
        loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        optimizer.step()

        current_lr = optimizer.param_groups[0]['lr']
        lrs.append(current_lr)
        losses.append(loss.item())

        for pg in optimizer.param_groups:
            pg['lr'] *= ratio

        if math.isnan(loss.item()) or loss.item() > 10 * losses[0]:
            break

    smoothed, alpha = [], 0.1
    running = losses[0]
    for l in losses:
        running = alpha * l + (1 - alpha) * running
        smoothed.append(running)

    plt.figure(figsize=(8, 4))
    plt.semilogx(lrs, smoothed)
    plt.xlabel('Learning Rate (log scale)')
    plt.ylabel('Loss (EMA smoothed)')
    plt.title('LR Range Test — pick LR just before the upturn')
    plt.grid(True, which='both', alpha=0.3)
    plt.show()
    return lrs, losses

### Practical LR defaults

When a range test is impractical (limited compute, tight deadline), empirically validated defaults:

| Setting | Typical peak LR |
|---|---|
| Pretraining (125M–1B params) | 3e-4 – 6e-4 |
| Pretraining (7B+ params) | 1e-4 – 3e-4 |
| Full fine-tuning (any size) | 1e-5 – 5e-5 |
| LoRA fine-tuning (r=8–64) | 1e-4 – 3e-4 |
| DPO / GRPO | 1e-6 – 5e-6 |

### Why fine-tuning LR matters more than pretraining LR

Pretraining loss is relatively forgiving across a wide LR range — the model will converge (slowly or quickly) across more than an order of magnitude. Fine-tuning is far more sensitive:

- **Catastrophic forgetting** — gradient updates overwrite learned representations. A large LR destroys general capabilities faster than the fine-tuning task can rebuild them.[^forgetting]
- Fine-tuning dataset is small: large LR causes overfitting to noise in the first few steps.
- With LoRA, adapter weights start near zero. A large LR makes them grow too fast, dominating the pretrained weights before they have learned anything useful.

[^forgetting]: Catastrophic forgetting manifests as a model that scores high on the fine-tuning task but completely fails at adjacent tasks it could previously handle. It is especially severe when fine-tuning on narrow instruction datasets without mixing in general data.

In [ ]:
#| code-fold: true
def measure_catastrophic_forgetting(
    model,
    original_eval_loader,
    fine_tune_loader,
    lr: float,
    num_steps: int = 200,
):
    """
    Fine-tunes model for `num_steps` steps at `lr` and measures
    how much performance on the original task degrades.

    Returns:
        original_losses: loss on original_eval_loader before fine-tuning
        post_ft_losses:  loss on original_eval_loader after fine-tuning
    """
    import copy
    model_ft = copy.deepcopy(model)
    model_ft.train()
    opt = torch.optim.AdamW(model_ft.parameters(), lr=lr)

    # Baseline evaluation on original task
    model_ft.eval()
    original_losses = []
    with torch.no_grad():
        for x, y in original_eval_loader:
            logits, loss = model_ft(x, y)
            original_losses.append(loss.item())

    # Fine-tune
    model_ft.train()
    data_iter = iter(fine_tune_loader)
    for step in range(num_steps):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(fine_tune_loader)
            x, y = next(data_iter)
        opt.zero_grad()
        _, loss = model_ft(x, y)
        loss.backward()
        opt.step()

    # Post fine-tune evaluation on original task
    model_ft.eval()
    post_ft_losses = []
    with torch.no_grad():
        for x, y in original_eval_loader:
            logits, loss = model_ft(x, y)
            post_ft_losses.append(loss.item())

    print(f"Original task loss — before: {sum(original_losses)/len(original_losses):.4f}")
    print(f"Original task loss — after:  {sum(post_ft_losses)/len(post_ft_losses):.4f}")
    return original_losses, post_ft_losses

## Gradient Accumulation

GPU memory limits the physical batch size. For pretraining we want 256K–2M tokens per step, but a single GPU may only fit 8K tokens in memory at once. **Gradient accumulation** bridges the gap: we split the target batch into $k$ micro-batches, run forward and backward for each, let PyTorch accumulate the gradients, then call `optimizer.step()` once:

$$\text{effective\_batch\_size} = \text{micro\_batch\_size} \times k \times n_\text{GPU}$$

### Why you must divide by $k$

If we simply call `.backward()` on each micro-batch's loss and sum, the accumulated gradient is $k\times$ larger than the single-batch gradient. This effectively multiplies the learning rate by $k$: at $k=32$, what was a stable `lr=3e-4` becomes an unstable `lr=9.6e-3`. The fix is to divide each micro-batch loss by $k$ before calling `.backward()`:

$$\text{loss per micro-step} = \frac{\mathcal{L}(\mathcal{B}_i)}{k}$$

The correct pattern:

In [ ]:
def train_step_with_accumulation(model, optimizer, get_batch, accum_steps: int):
    optimizer.zero_grad()                    # zero ONCE before the loop  # <1>
    total_loss = 0.0

    for micro_step in range(accum_steps):
        x, y = get_batch()
        logits, loss = model(x, y)
        (loss / accum_steps).backward()      # scale before backward  # <2>
        total_loss += loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # <3>
    optimizer.step()
    return total_loss / accum_steps

1. `zero_grad` once before the loop — gradients accumulate across all micro-steps. Moving it inside the loop is the most common bug in gradient accumulation code.
2. Divide by `accum_steps` before `backward()` so the summed gradient equals the true full-batch gradient.
3. Clip the full accumulated gradient, not per micro-step. Clipping per step is too aggressive and prevents the model from learning from large-gradient examples.

:::{.callout-caution}
## The silent `zero_grad` bug

If `zero_grad` is inside the loop, gradients are wiped between micro-steps — only the last micro-step's gradient survives. The loss looks reasonable; training appears to proceed. The bug is completely silent until you notice that effective batch size is `micro_batch_size`, not `micro_batch_size × accum_steps`.

:::

In [ ]:
# BUG: zero_grad inside the loop
for micro_step in range(accum_steps):
    optimizer.zero_grad()   # ← WRONG: discards all previous accumulation
    x, y = get_batch()
    _, loss = model(x, y)
    (loss / accum_steps).backward()
optimizer.step()
# Result: equivalent to batch_size, not batch_size * accum_steps

### Gradient accumulation with mixed precision

When combining gradient accumulation with `torch.autocast`, the context manager belongs [inside]{.underline} the accumulation loop, around the forward pass only. The backward pass and optimizer step always run in FP32 regardless:

In [ ]:
optimizer.zero_grad()
for micro_step in range(accum_steps):
    x, y = get_batch()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits, loss = model(x, y)
    (loss / accum_steps).backward()  # backward outside autocast is fine  # <1>

optimizer.step()

1. `backward()` always runs in FP32 regardless of `autocast`. The context manager only affects the forward pass — specifically which ops are cast to lower precision.

:::{.callout-note}
## Variable-length sequence normalization

For language models, cross-entropy is averaged over tokens. When sequences have different lengths (or masked positions), dividing by $k$ is not equivalent to the full-batch gradient. The correct normalization divides by the total token count $N = \sum_a N_a$ across all micro-batches:

```python
IGNORE_IDX = -100
m = 0
for i, (x, y) in enumerate(train_loader):
    outs = model(x)
    loss = F.cross_entropy(outs, y, reduction="sum")
    loss.backward()
    m += (y != IGNORE_IDX).int().sum()

    if (i + 1) % S == 0:
        for p in model.parameters():
            p.grad /= m           # normalize by true token count
        optimizer.step()
        optimizer.zero_grad()
        m = 0
```

Using a fixed divisor $S$ when token counts vary creates a silent scaling error proportional to the average length mismatch.

:::

In [ ]:
#| code-fold: true
# Verify gradient accumulation == true large batch gradient
# True large batch (batch_size=32)
x_large, y_large = get_batch(batch_size=32)
model.zero_grad()
_, loss_true = model(x_large, y_large)
loss_true.backward()
true_grad_norm = sum(p.grad.norm()**2 for p in model.parameters()
                     if p.grad is not None).sqrt().item()

# Accumulated (4 micro-batches of 8)
model.zero_grad()
for i in range(4):
    x_micro = x_large[i*8:(i+1)*8]
    y_micro = y_large[i*8:(i+1)*8]
    _, loss_micro = model(x_micro, y_micro)
    (loss_micro / 4).backward()
accum_grad_norm = sum(p.grad.norm()**2 for p in model.parameters()
                      if p.grad is not None).sqrt().item()

print(f"True large-batch grad norm:  {true_grad_norm:.6f}")
print(f"Accumulated grad norm:        {accum_grad_norm:.6f}")
# Should be approximately equal

## Mixed Precision — BF16 and FP16

FP32 is 4 bytes per value. A 100M-parameter model's activations fill roughly 4 GB for a single forward pass at batch size 32. BF16 halves this. Additionally, NVIDIA Tensor Cores natively accelerate BF16/FP16 matrix multiplications: 2–4× faster than FP32 on A100-class GPUs.

**Mixed precision** (not pure BF16 training) means: run the forward pass and backward pass in lower precision where it is safe; keep the optimizer state and master weights in FP32.

### FP16 vs BF16

| Property | FP16 | BF16 |
|---|---|---|
| Total bits | 16 | 16 |
| Exponent bits | 5 | 8 (same as FP32) |
| Mantissa bits | 10 | 7 |
| Max value | 65504 | ~3.4 × 10³⁸ |
| Min positive normal | ~6.1 × 10⁻⁵ | ~1.2 × 10⁻³⁸ |
| Overflow risk | High | None (same range as FP32) |
| Precision | Higher | Lower |
| Typical use | A100, consumer GPUs | A100/H100 — preferred for LLMs |

The critical issue with FP16: its maximum value is 65504. During the forward pass, intermediate activations — especially logits before softmax normalization — can easily exceed this and produce `inf`, which immediately propagates to `nan` gradients. The fix is `GradScaler`, which multiplies the loss by a large factor before `backward()` (scaling gradients up to avoid underflow near FP16's minimum), then divides by the same factor before the optimizer step.

BF16 eliminates this entirely: its 8-bit exponent gives it the same dynamic range as FP32 — no overflow, no `GradScaler` needed.

In [ ]:
#| code-fold: true
import torch

x_fp16 = torch.tensor([60000.0], dtype=torch.float16)
print(f"FP16 value:        {x_fp16.item()}")
print(f"FP16 squared:      {(x_fp16 * x_fp16).item()}")   # overflows to inf

x_bf16 = torch.tensor([60000.0], dtype=torch.bfloat16)
print(f"BF16 value:        {x_bf16.item()}")
print(f"BF16 squared:      {(x_bf16 * x_bf16).item()}")   # fine

x_bf16_large = torch.tensor([1e38], dtype=torch.bfloat16)
print(f"BF16 1e38:         {x_bf16_large.item()}")         # fine, same range as FP32

### `torch.autocast`

`torch.autocast` is a context manager that automatically casts operations to lower precision. Within the context, matmuls and convolutions use BF16/FP16; operations sensitive to precision — softmax, layer norm, loss computation — remain in FP32. The casting decisions are made per-operation based on a built-in policy; we do not need to manually cast tensors.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# BF16 training (recommended for modern GPUs)
with torch.autocast(device_type=device.type, dtype=dtype):
    logits, loss = model(x, y)
loss.backward()    # backward always runs in FP32

# FP16 training requires GradScaler
scaler = torch.cuda.amp.GradScaler()
with torch.autocast(device_type='cuda', dtype=torch.float16):
    logits, loss = model(x, y)
scaler.scale(loss).backward()   # scaled backward
scaler.step(optimizer)          # unscale before step
scaler.update()                 # update scale factor

:::{.callout-note}
On recent GPUs (A100, H100, RTX 4000-series), prefer BF16. It eliminates the entire `GradScaler` machinery while maintaining the same memory and throughput benefits as FP16.

:::

## Learning Rate Schedule — Cosine with Warmup

### Why warmup?

Adam's bias correction at step $t = 1$: $\hat{m}_1 = m_1 / (1 - \beta_1) = g_1 / 0.1 = 10 g_1.$ The first gradient is amplified roughly 10×. With `lr=3e-4`, the effective first step is `3e-3` — ten times larger than intended. [Warmup keeps early steps small and prevents this instability.]{.mark}

During the warmup phase, we linearly ramp the LR from $0$ to the peak value over a fixed number of steps. Warmup duration is typically 1–2% of total steps. Too long delays learning; too short causes early instability.

### Cosine decay

After warmup, the LR follows a cosine curve from the peak down to a floor:

$$\text{lr}(t) = \text{min\_lr} + \frac{1}{2}(\text{max\_lr} - \text{min\_lr})\left(1 + \cos\left(\frac{\pi (t - t_\text{warmup})}{T - t_\text{warmup}}\right)\right)$$

The floor is set to `min_lr = 0.1 × max_lr` — the Chinchilla convention. Decaying all the way to zero wastes the final phase of training on near-zero updates; keeping `min_lr` at 10% of peak preserves meaningful progress to the end.

We implement this as a `LambdaLR` scheduler:

In [ ]:
import math
from torch.optim.lr_scheduler import LambdaLR

def make_cosine_schedule(
    optimizer,
    max_lr:       float,
    min_lr:       float,
    warmup_steps: int,
    total_steps:  int,
) -> LambdaLR:
    """Cosine decay with linear warmup.  # <1>

    Returns a LambdaLR scheduler that can be stepped once per optimizer step.
    """
    min_ratio = min_lr / max_lr       # relative floor                     # <2>

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:                                            # <3>
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        progress = min(progress, 1.0)                                      # <4>
        return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    return LambdaLR(optimizer, lr_lambda)

1. `LambdaLR` calls `lr_lambda(step)` and multiplies `base_lr` by the result. We set `base_lr = max_lr` when constructing the optimizer.
2. Express the minimum LR as a ratio so the lambda is scale-independent — the same function works regardless of `max_lr`.
3. Linear warmup: LR rises from 0 to `max_lr` over `warmup_steps`.
4. Clamp `progress` at 1.0 to freeze LR at `min_lr` if training extends beyond `total_steps`.

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt
import torch

dummy_optimizer = torch.optim.AdamW([torch.nn.Parameter(torch.zeros(1))], lr=3e-4)
scheduler = make_cosine_schedule(dummy_optimizer, max_lr=3e-4, min_lr=3e-5,
                                  warmup_steps=100, total_steps=5000)

lrs = []
for step in range(5000):
    lrs.append(dummy_optimizer.param_groups[0]['lr'])
    scheduler.step()

plt.figure(figsize=(10, 3))
plt.plot(lrs)
plt.axvline(100, color='gray', linestyle='--', label='end of warmup')
plt.axhline(3e-5, color='orange', linestyle='--', label='min_lr')
plt.xlabel('step')
plt.ylabel('learning rate')
plt.title('Cosine schedule with linear warmup')
plt.legend()
plt.tight_layout(); plt.show()

## Chinchilla Scaling Laws

How many training tokens are needed for a model of $N$ parameters? Hoffmann et al. (2022) fit an empirical loss model:

$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$

where $N$ is the number of parameters, $D$ is the number of training tokens, and $E, A, B, \alpha, \beta$ are fitted constants. The compute-optimal training point — where neither the model nor the data is the binding constraint — gives the famous rule:

$$D^* \approx 20N$$

At this optimal point, we allocate roughly equal compute to making the model larger and to showing it more data. Pre-Chinchilla models (GPT-3, for example) were significantly undertrained: a 175B model trained on 300B tokens has $D/N \approx 1.7$, far below the optimal 20.

### FLOPs per token

For a transformer with $N$ non-embedding parameters, the FLOPs per training token are approximately:

$$\text{FLOPs per training token} \approx 6N$$

This comes from 2× for the forward pass (two multiplications per weight in a matmul) and 4× for the backward pass (gradients with respect to inputs and weights). We use this to convert a hardware budget into a token budget:

In [ ]:
def chinchilla_optimal(total_flops: float) -> dict:
    """Compute-optimal N and D given a FLOPs budget (Hoffmann et al. 2022)."""
    # Chinchilla formula: N* ≈ (C / (20 * 6))^0.5 ≈ C^0.5 / 10.95
    optimal_params = (total_flops / (20 * 6)) ** 0.5
    optimal_tokens = 20 * optimal_params
    return {
        'optimal_params_M': optimal_params / 1e6,
        'optimal_tokens_B': optimal_tokens / 1e9,
        'optimal_params':   optimal_params,
        'optimal_tokens':   optimal_tokens,
    }


def flops_per_step(n_params: int, batch_tokens: int) -> float:
    """Approximate FLOPs for one optimizer step."""
    return 6 * n_params * batch_tokens


def plan_training_run(
    n_params:        int,
    gpu_flops_per_s: float,
    gpu_count:       int,
    hours:           float,
    batch_tokens:    int,
) -> dict:
    """Print a training plan and return key metrics."""
    total_flops  = gpu_flops_per_s * gpu_count * hours * 3600
    total_steps  = int(total_flops / flops_per_step(n_params, batch_tokens))
    total_tokens = total_steps * batch_tokens

    chinchilla   = chinchilla_optimal(total_flops)
    token_ratio  = total_tokens / chinchilla['optimal_tokens']

    print(f"\nTraining Run Plan")
    print(f"{'─'*50}")
    print(f"  Model parameters:    {n_params/1e6:.1f}M")
    print(f"  Hardware:            {gpu_count}× GPU @ {gpu_flops_per_s/1e12:.0f} TFLOP/s")
    print(f"  Time budget:         {hours:.1f} hours")
    print(f"  Total FLOPs:         {total_flops/1e18:.2f} EFLOPs")
    print(f"  Total steps:         {total_steps:,}")
    print(f"  Total tokens:        {total_tokens/1e6:.0f}M")
    print(f"{'─'*50}")
    print(f"  Chinchilla-optimal N: {chinchilla['optimal_params_M']:.1f}M params")
    print(f"  Chinchilla-optimal D: {chinchilla['optimal_tokens_B']:.2f}B tokens")
    print(f"  Our token ratio:      {token_ratio:.2f}× optimal")
    if token_ratio < 0.1:
        print(f"  ⚠ Severely undertrained — consider smaller model")
    elif token_ratio < 0.5:
        print(f"  ⚠ Undertrained — model could learn more with more data")
    elif token_ratio > 2.0:
        print(f"  ⚠ Overtrained on this dataset — model may be memorizing")
    else:
        print(f"  ✓ Near compute-optimal")

    return {
        'total_steps':  total_steps,
        'total_tokens': total_tokens,
        'token_ratio':  token_ratio,
    }

Our nano model (29.9M parameters) on a consumer GPU:

In [ ]:
# Our nano model on a consumer GPU
plan = plan_training_run(
    n_params=29_900_000,    # ~29.9M nano GPT (with SwiGLU)
    gpu_flops_per_s=20e12,  # RTX 3090: ~20 TFLOP/s (BF16)
    gpu_count=1,
    hours=1.0,              # 1 hour of training
    batch_tokens=8 * 256,   # batch_size=8, seq_len=256 → 2048 tokens/step
)

The nano model (29.9M parameters) is Chinchilla-optimal at $D^* = 20 \times 29.9\text{M} = 598\text{M}$ tokens. TinyShakespeare has only ~1M tokens — about 0.2% of the optimal amount. The training runs in this series demonstrate mechanics and code structure, not Chinchilla-optimal generalization. [For a compute-optimal run, use a much larger corpus.]{.mark}

## Pretraining vs Fine-Tuning Hyperparameters

The right hyperparameters differ substantially between pretraining and fine-tuning. The core reason: pretraining explores a fresh random initialization over billions of tokens; fine-tuning makes small corrections to a converged model over thousands.

| Hyperparameter | Pretraining | Fine-tuning (Full) | Fine-tuning (LoRA) |
|---|---|---|---|
| Optimizer | AdamW | AdamW | AdamW |
| Peak LR | 3e-4 (125M), 1e-4 (1B+) | 1e-5 – 5e-5 | 1e-4 – 3e-4 |
| $\beta_1$ | 0.9 | 0.9 | 0.9 |
| $\beta_2$ | 0.95 | 0.999 | 0.999 |
| Weight decay | 0.1 | 0.01 – 0.1 | 0.01 |
| Warmup steps | 1–2% of total | 3–10% of total | 10% of total |
| Grad clip | 1.0 | 1.0 | 1.0 |
| Effective batch (tokens) | 256K – 4M | 8K – 128K | 4K – 32K |
| Accum steps | 4–64 | 1–8 | 1–4 |

**$\beta_2$ changes.** Pretraining uses 0.95 (memory ~20 steps) for responsiveness over millions of steps where gradient magnitude distributions shift. Fine-tuning uses 0.999 (memory ~1000 steps) for smoother, more stable second-moment estimates on a much shorter run.

**Weight decay is lower for fine-tuning.** High weight decay fights the gradient update — the model adapts while decay continuously pulls weights toward zero. For LoRA adapters starting near zero, even 0.01 can slow convergence if the adapter is small (low rank).

**Layer-wise LR for full fine-tuning.** Early layers capture general features (syntax, semantics) learned over the entire pretraining corpus; later layers are more task-specific. Applying the same LR to all layers is aggressive: early layers should stay close to their pretrained values, later layers have more room to adapt.

In [ ]:
def make_layerwise_optimizer(model, base_lr, num_layers):
    """
    Linear LR decay: layer 0 gets base_lr * 0.1, last layer gets base_lr.
    Forces early layers to stay close to pretrained representations.
    """
    param_groups = []
    for i, (name, p) in enumerate(model.named_parameters()):
        layer_idx = 0
        for j in range(num_layers):
            if f'layers.{j}.' in name or f'layer.{j}.' in name:
                layer_idx = j + 1
                break
        lr_scale = 0.1 + 0.9 * (layer_idx / num_layers)   # 0.1 → 1.0
        param_groups.append({
            'params': [p],
            'lr':     base_lr * lr_scale,
            'weight_decay': 0.01 if p.ndim >= 2 else 0.0,
        })
    return torch.optim.AdamW(param_groups, lr=base_lr, betas=(0.9, 0.999))

Putting it all together — an `OptimizerConfig` dataclass and a `build_optimizer` function that handles both pretraining and fine-tuning configurations:

In [ ]:
from dataclasses import dataclass
from torch.optim.lr_scheduler import LambdaLR
import math


@dataclass
class OptimizerConfig:
    # Core
    lr:             float = 2e-5
    weight_decay:   float = 0.01
    betas:          tuple = (0.9, 0.999)
    eps:            float = 1e-8
    grad_clip:      float = 1.0

    # Schedule
    warmup_ratio:   float = 0.06      # fraction of steps for warmup
    min_lr_ratio:   float = 0.1       # min_lr = lr * min_lr_ratio

    # Accumulation
    accum_steps:    int   = 1

    # LoRA-specific
    lora_lr_scale:  float = 1.0


def build_optimizer(model: nn.Module, config: OptimizerConfig, total_steps: int):
    """Returns (optimizer, scheduler) ready for a fine-tuning run."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2:
            decay.append(p)
        else:
            no_decay.append(p)

    optimizer = torch.optim.AdamW(
        [{'params': decay,    'weight_decay': config.weight_decay},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=config.lr,
        betas=config.betas,
        eps=config.eps,
    )

    warmup_steps = int(total_steps * config.warmup_ratio)
    min_ratio    = config.min_lr_ratio

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        progress = min(progress, 1.0)
        return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    return optimizer, LambdaLR(optimizer, lr_lambda)


def fine_tune_step(model, optimizer, scheduler, get_micro_batch,
                   accum_steps: int, grad_clip: float) -> float:
    """One optimizer step with gradient accumulation and autocast."""
    optimizer.zero_grad()
    total_loss = 0.0
    for _ in range(accum_steps):
        x, y = get_micro_batch()
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits, loss = model(x, y)
        (loss / accum_steps).backward()
        total_loss += loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    scheduler.step()
    return total_loss / accum_steps

## Summary

| Concept | Key detail |
|---|---|
| FP16 overflow | Max value 65504 — activations can exceed this. Requires `GradScaler`. |
| BF16 dynamic range | Same 8-bit exponent as FP32 — never overflows. No `GradScaler` needed. |
| `torch.autocast` | Casts matmul/conv to BF16; sensitive ops (softmax, RMSNorm) stay FP32. |
| Adam warmup reason | Bias correction at $t=1$ amplifies first gradient 10×. Warmup keeps early steps small. |
| Warmup duration | 1–2% of total steps. Too long delays learning; too short causes early instability. |
| Cosine decay | $\text{min\_lr} + \frac{1}{2}(\text{max\_lr} - \text{min\_lr})(1 + \cos(\pi t / T))$ |
| `min_lr` choice | 10% of `max_lr` — Chinchilla convention. Keeps updates meaningful to the end. |
| Chinchilla $D^* = 20N$ | Compute-optimal training: 20 tokens per parameter. |
| FLOPs per token | $\approx 6N$ per training token (2× forward, 4× backward). |
| Decay vs no-decay params | Weight matrices: weight decay. Biases + RMSNorm scales: no weight decay. |
| Gradient accumulation | Divide by $k$; `zero_grad` once before the loop; wrap `autocast` per micro-step. |

:::{.callout-note}
## References

- [`nanoGPT/train.py`](https://github.com/karpathy/nanoGPT/blob/master/train.py) — `get_lr()`, `configure_optimizers()`: the original cosine schedule implementation and AdamW parameter group construction.
- Hoffmann et al. (2022), "Training Compute-Optimal Large Language Models" (Chinchilla) — scaling law constants $D^* \approx 20N.$

:::

## Exercises

**1.** Run the cosine schedule visualization with four configurations: `warmup_steps` ∈ {10, 100, 500} and `min_lr` ∈ {0, 0.1×max, 0.5×max}. Run 500 training steps for each and plot the loss curve. Confirm that too little warmup causes instability and `min_lr=0` causes the model to plateau earlier.

**2.** Verify the gradient accumulation equivalence: train for 100 steps with `batch_size=32, accumulation=1` and separately with `batch_size=8, accumulation=4`. Both have effective batch size 32. Compare the final loss and gradient norm trace. They should be nearly identical.

**3.** Use `plan_training_run` to compute the Chinchilla-optimal training duration for the nano model on your machine. Then run for exactly that many steps and compare the final eval loss to a run that uses 5× more steps (overtrained). Confirm that overtraining on TinyShakespeare causes the eval/train gap to widen.

**4.** Implement LR restart (SGDR): after each full cosine cycle, reset LR to `max_lr` and start a new cosine decay with period 2× the previous period. Compare loss curves between standard cosine and SGDR on 5000 training steps.

**5.** Implement the variable-length sequence gradient accumulation correction. Train on a dataset with mixed sequence lengths and compare loss curves between the naive $k$-divisor and the correct total-token-count divisor.

■